# 🟡 Solution: Point in Polygon

**Primitive:** ray casting, broadcasting over `(P,1)` vs `(1,V)` arrays

**Reduction:** `out[p]` is True iff the rightward ray from `points[p]` crosses an odd number of polygon edges. For each edge, straddle-check and compute crossing x via vectorised division.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# primitive: ray casting (broadcasting over P points and V edges)

import numpy as np

def point_in_polygon(points, polygon):
    points  = np.asarray(points,  dtype=float)
    polygon = np.asarray(polygon, dtype=float)
    px = points[:, 0:1]  # (P, 1)
    py = points[:, 1:2]  # (P, 1)
    v1 = polygon          # (V, 2)
    v2 = np.roll(polygon, -1, axis=0)
    x1, y1 = v1[:, 0], v1[:, 1]  # (V,)
    x2, y2 = v2[:, 0], v2[:, 1]  # (V,)
    # Edge straddles the horizontal ray?
    straddles = (y1 > py) != (y2 > py)           # (P, V)
    # x coord of crossing; safe because straddles==True ⟹ y2-y1 != 0
    denom = np.where(y2 - y1 == 0, 1.0, y2 - y1)
    x_cross = x1 + (py - y1) / denom * (x2 - x1)  # (P, V)
    crosses = straddles & (px < x_cross)
    return np.sum(crosses, axis=1) % 2 == 1

In [ ]:
# 🔍 Verify solution
square = np.array([[0.,0.],[1.,0.],[1.,1.],[0.,1.]])
pts = np.array([[0.5, 0.5],   # center — inside
                [2.0, 0.5],   # right — outside
                [-0.5, 0.5]]) # left — outside
print(point_in_polygon(pts, square))  # expect [True, False, False]

# Concave L-shape
L = np.array([[0.,0.],[3.,0.],[3.,2.],[2.,2.],[2.,3.],[0.,3.]])
test_pts = np.array([[1.,1.], [2.5,2.5], [4.,1.]])
print(point_in_polygon(test_pts, L))  # expect [True, False, False]

In [ ]:
# ✅ Inline test suite
import numpy as np, time

# ── Test 1: unit square ────────────────────────────────────────────────────
sq = np.array([[0.,0.],[1.,0.],[1.,1.],[0.,1.]])
pts = np.array([[0.5,0.5],[2.,0.5],[-0.5,0.5]])
r = point_in_polygon(pts, sq)
assert r[0]==True and r[1]==False and r[2]==False, f"Square test: {r}"
print("Test 1 passed: basic square")

# ── Test 2: concave L-shape ────────────────────────────────────────────────
L = np.array([[0.,0.],[3.,0.],[3.,2.],[2.,2.],[2.,3.],[0.,3.]])
pts2 = np.array([[1.,1.],[2.5,2.5],[4.,1.]])
r2 = point_in_polygon(pts2, L)
assert r2[0]==True,  f"Inside L: {r2[0]}"
assert r2[1]==False, f"Notch of L: {r2[1]}"
assert r2[2]==False, f"Outside L: {r2[2]}"
print("Test 2 passed: concave L-shape")

# ── Test 3: triangle ───────────────────────────────────────────────────────
tri = np.array([[0.,0.],[4.,0.],[2.,3.464]])
pts3 = np.array([[2.,1.],[2.,-1.],[5.,1.]])
r3 = point_in_polygon(pts3, tri)
assert r3[0]==True and r3[1]==False and r3[2]==False, f"Triangle: {r3}"
print("Test 3 passed: triangle")

# ── Test 4: all outside ─────────────────────────────────────────────────────
tiny = np.array([[10.,10.],[10.1,10.],[10.1,10.1],[10.,10.1]])
r4 = point_in_polygon(np.zeros((5,2)), tiny)
assert not r4.any(), f"All outside: {r4}"
print("Test 4 passed: all outside")

# ── Test 5: P=5000 large polygon ───────────────────────────────────────────
rng=np.random.default_rng(42); n=60
a=np.linspace(0,2*np.pi,n,endpoint=False)
poly = np.stack([np.cos(a), np.sin(a)], axis=1)
pts5 = rng.uniform(-1.5,1.5,(5000,2))
t0=time.time(); r5=point_in_polygon(pts5, poly); elapsed=time.time()-t0
dists = np.hypot(pts5[:,0], pts5[:,1])
assert r5[dists<0.95].all(), "Points clearly inside should be True"
assert not r5[dists>1.05].any(), "Points clearly outside should be False"
assert elapsed < 3.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: P=5000 ({elapsed:.3f}s)")

print("\nAll tests passed!")